# Prototipo: clustering y rutas de telemetria GPS (Artisa)

Notebook de experimentacion libre: acá se prueba y se valida la lógica de ML con datos reales, sin la obligación de mantenerse idéntico a ningún archivo del proyecto. Cuando algo quede bueno y probado, recién ahí se porta a `modelo_machine/` (no tiene que ser copy-paste literal, solo la misma idea ya validada).

**Pipeline real del proyecto:** MongoDB (`gps_data.raw_gps`) -> ETL -> MariaDB (`raw_gps`) -> `modelo_machine` lee de MariaDB.

**Aqui:** corre local en VS Code (kernel Jupyter/ipykernel), leyendo una muestra exportada a CSV en vez de conectarse directo a Mongo/MariaDB. Tambien puedes subirlo a Google Colab sin cambios (usa `files.upload()` ahi en vez de leer el CSV local).

## Hoja de ruta ML — estado actual

Leyenda: ✅ hecho · 🟡 parcial (falta robustecer) · ⬜ pendiente

Los titulos "Paso N" de este notebook coinciden con el numero de esta tabla.

| # | Paso | Estado | Donde esta / que falta |
|---|------|--------|------------------------|
| 1 | Definir problema y objetivo | 🟡 | El Paso 7 dice "zonas frecuentes", pero falta definir el objetivo de negocio y como medir el exito |
| 2 | Recoleccion de datos | ✅ | Seccion Paso 2 (MariaDB). Ojo: solo toma las ultimas `MAX_ROWS` filas, revisar representatividad |
| 3 | Analisis exploratorio (EDA) | ⬜ | Solo hay un scatter. Faltan distribuciones, nulos, outliers, correlaciones, reportes por vehiculo/hora |
| 4 | Limpieza y preprocesamiento | 🟡 | Seccion Paso 4 y 5: `clean_rows`, `impute_missing`, `scale_numeric`. Falta filtrar coordenadas invalidas/duplicadas y outliers de lat/lon |
| 5 | Ingenieria de caracteristicas | 🟡 | Seccion Paso 4 y 5: `hora_dia`, `exceso_velocidad`, one-hot de `event_name`. Faltan ventanas de tiempo, dia de la semana, agregaciones por vehiculo |
| 6 | Division train / validation / test | ⬜ | Se entrena con todos los datos. Dividir cronologicamente para evitar fuga de datos |
| 7 | Seleccion y entrenamiento de modelos | 🟡 | Seccion Paso 7: solo DBSCAN. Falta un modelo base y comparar KMeans, HDBSCAN, GMM |
| 8 | Evaluacion con metricas | 🟡 | Solo `cobertura_pct` y `puntos_ruido`. Faltan silhouette, Davies-Bouldin, Calinski-Harabasz |
| 9 | Validacion y ajuste de hiperparametros | ⬜ | `radius_km` / `min_samples` se ajustan a mano. Falta busqueda sistematica (grid / Optuna) y prueba de estabilidad |
| 10 | Interpretacion de resultados | 🟡 | Seccion Paso 10: tabla de zonas, scatter, mapa folium y geocercas de faenas (poligonos reales). Falta perfilar cada zona (horarios, exceso de velocidad, tipo de evento) |
| 11 | Despliegue | 🟡 | Seccion Paso 11: ya existe el servicio en `modelo_machine`. Falta guardar el pipeline/modelo serializado |
| 12 | Monitoreo y mantenimiento | ⬜ | Sin deteccion de data drift, reentrenamiento programado ni versionado de modelos |

Ir actualizando esta tabla a medida que se completen los pasos.

## Paso 2 — Recolección de datos (conexión a MariaDB)

Sin CSV ni Mongo: la celda de abajo lee la tabla `raw_gps` de MariaDB, que es donde el ETL ya dejo los datos (igual que hace `modelo_machine`).

Requisitos:

1. `artisa-mariadb` corriendo y `healthy` (`docker compose ps`). El puerto 3306 ya esta publicado en `127.0.0.1` en `compose.yaml`.
2. Las credenciales salen del `.env` de la raiz (`MARIADB_USER`, `MARIADB_PASSWORD`, `MARIADB_DATABASE`).
3. `pymysql` instalado en el `.venv`.
4. Si sale `Access denied`, la contraseña del `.env` no coincide con la que tiene el volumen de MariaDB; hay que corregirla antes de seguir.

In [ ]:
# Si falta algun paquete en tu entorno local: pip install pymysql pandas numpy scikit-learn matplotlib

import json
import statistics
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql
from sklearn.cluster import DBSCAN

In [ ]:
# --- Parametros editables: rango de fechas a traer ---
FECHA_DESDE = "2026-04-01"   # fecha de inicio (YYYY-MM-DD)
RANGO_CANTIDAD = 1            # cuantas unidades de RANGO_UNIDAD
RANGO_UNIDAD = "semana"       # "dia", "semana" o "mes"

_OFFSET_POR_UNIDAD = {
    "dia": lambda n: pd.DateOffset(days=n),
    "semana": lambda n: pd.DateOffset(weeks=n),
    "mes": lambda n: pd.DateOffset(months=n),
}

fecha_desde = pd.Timestamp(FECHA_DESDE)
fecha_hasta = fecha_desde + _OFFSET_POR_UNIDAD[RANGO_UNIDAD](RANGO_CANTIDAD)


def load_env(path):
    """Parser minimo de .env (KEY=VALUE), sin dependencias extra."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env = load_env(Path("..") / ".env")  # credenciales de MariaDB

COLUMNAS = ["report_id", "asset_id", "asset_name", "report_date", "latitude", "longitude",
            "speed", "cartography_limit_speed", "event_name"]

with pymysql.connect(
    host="127.0.0.1", port=3306,
    user=env["MARIADB_USER"], password=env["MARIADB_PASSWORD"], database=env["MARIADB_DATABASE"],
    charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor,
    read_timeout=180,
) as connection, connection.cursor() as cursor:
    # Solo vehiculos clasificados como 'camion' (deja afuera camionetas, minibus, etc.)
    cursor.execute(
        """SELECT v.asset_id FROM vehiculos v
           JOIN tipos_vehiculo t ON t.id_tipo = v.id_tipo
           WHERE t.nombre = 'camion' ORDER BY v.asset_id"""
    )
    camiones = [row["asset_id"] for row in cursor.fetchall()]

    # Un SELECT por asset_id (usa el indice idx_raw_gps_asset_fecha): un JOIN/GROUP BY
    # global sobre raw_gps se cuelga con el volumen actual de la tabla (12M+ filas).
    filas = []
    for asset_id in camiones:
        cursor.execute(
            f"""SELECT {', '.join(COLUMNAS)} FROM raw_gps USE INDEX (idx_raw_gps_asset_fecha)
                WHERE asset_id=%s AND report_date BETWEEN %s AND %s ORDER BY report_date""",
            (asset_id, fecha_desde.to_pydatetime(), fecha_hasta.to_pydatetime()),
        )
        filas.extend(cursor.fetchall())

df = pd.DataFrame(filas, columns=COLUMNAS)
df["report_date"] = pd.to_datetime(df["report_date"])

print(f"{len(df):,} reportes de {df['asset_id'].nunique()} camiones "
      f"({fecha_desde.date()} -> {fecha_hasta.date()})")
df.head()

## Paso 4 y 5 — Limpieza / preprocesamiento e ingeniería de características

5 funciones **vectorizadas con pandas** (reciben/devuelven un DataFrame). La primera versión (fila por fila con loops en Python) se colgaba a partir de ~180.000 filas (10 camiones); esta corre sobre los 733.005 datos de los 40 camiones sin problema.

- **Paso 4 (limpieza y preprocesamiento):** `clean_rows`, `impute_missing`, `scale_numeric`.
- **Paso 5 (ingeniería de características):** `derive_features`, `encode_categorical`.

In [ ]:
NUMERIC_FEATURES = ("latitude", "longitude", "speed", "cartography_limit_speed",
                     "exceso_velocidad", "hora_dia")
SPEED_MIN_KMH = 0.0
SPEED_MAX_KMH = 200.0


def clean_rows(df, speed_min=SPEED_MIN_KMH, speed_max=SPEED_MAX_KMH):
    """Descarta reportes con velocidad fuera de un rango fisico razonable."""
    fuera_de_rango = df["speed"].notna() & ~df["speed"].between(speed_min, speed_max)
    return df.loc[~fuera_de_rango].reset_index(drop=True)


def impute_missing(df):
    """Rellena speed y cartography_limit_speed nulos con la mediana del lote."""
    df = df.copy()
    df["speed"] = df["speed"].fillna(df["speed"].median())
    df["cartography_limit_speed"] = df["cartography_limit_speed"].fillna(df["cartography_limit_speed"].median())
    return df


def derive_features(df):
    """Agrega hora_dia (0-23) y exceso_velocidad a cada fila."""
    df = df.copy()
    df["hora_dia"] = pd.to_datetime(df["report_date"]).dt.hour
    df["exceso_velocidad"] = df["speed"] - df["cartography_limit_speed"]
    return df


def encode_categorical(df, column="event_name"):
    """One-hot encoding dinamico: una columna nueva por cada categoria vista en el lote."""
    valores = df[column].fillna("desconocido")
    dummies = pd.get_dummies(valores, prefix=column, dtype=float)
    return pd.concat([df.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1), list(dummies.columns)


def scale_numeric(df, columns):
    """Estandarizacion z-score: (valor - media) / desviacion estandar."""
    df = df.copy()
    for columna in columns:
        media = df[columna].mean()
        desviacion = df[columna].std(ddof=0) or 1.0
        df[columna] = (df[columna] - media) / desviacion
    return df


def build_dataset(df, categorical_column="event_name"):
    """Orquesta limpieza -> imputacion -> features derivadas -> encoding -> escalado.

    Recibe y devuelve un DataFrame (vectorizado con pandas, no fila por fila).
    Devuelve (dataframe_transformado, columnas_numericas_finales).
    """
    df = clean_rows(df)
    df = impute_missing(df)
    df = derive_features(df)
    df, category_columns = encode_categorical(df, categorical_column)
    df = scale_numeric(df, list(NUMERIC_FEATURES))
    return df, list(NUMERIC_FEATURES) + category_columns

	Separar train / test

## Paso 4 y 5 (cont.) — Aplicar el pipeline y construir la matriz X

Convertimos el DataFrame a lista de dicts (lo que espera `build_dataset`), corremos el pipeline, y armamos la matriz `X` (numpy) lista para cualquier modelo.

In [ ]:
df_out, columnas = build_dataset(df)
X = df_out[columnas].to_numpy(dtype=float)
print("Matriz X:", X.shape, "columnas:", columnas)
X[:5]

## Paso 7 — Selección y entrenamiento del modelo: DBSCAN

Ojo: el clustering de zonas usa **lat/lon SIN escalar** (necesita coordenadas reales en radianes para la metrica haversine), no la matriz `X` del paso anterior. `X` queda lista para otros modelos futuros (deteccion de anomalias de velocidad, prediccion de ETA) que si se benefician del escalado.

**Muestreo:** a diferencia de `build_dataset` (Paso 4/5), este DBSCAN **si se cuelga con los 733.005 reportes completos** — los camiones detenidos generan vecindarios muy densos (miles de puntos casi identicos en el mismo lugar) y el arbol de busqueda (`ball_tree`) se degrada ahi. Se entrena con una muestra (`MAX_ROWS_DBSCAN`), que alcanza para detectar zonas frecuentes sin perder validez estadistica.

In [ ]:
EARTH_RADIUS_KM = 6371.0088
MAX_ROWS_DBSCAN = 20000  # mismo default que TELEMETRY_MODEL_MAX_ROWS en compose.yaml


def train_hotspots(rows, radius_km=0.8, min_samples=5):
    """Agrupa posiciones cercanas sin requerir rutas etiquetadas."""
    if len(rows) < max(10, min_samples * 2):
        raise ValueError("Se necesitan al menos 10 posiciones para entrenar telemetria")
    coordinates = np.radians([[float(row["latitude"]), float(row["longitude"])] for row in rows])
    labels = DBSCAN(
        eps=float(radius_km) / EARTH_RADIUS_KM,
        min_samples=int(min_samples),
        metric="haversine",
        algorithm="ball_tree",
    ).fit_predict(coordinates)

    grouped = defaultdict(list)
    noise = 0
    for row, label in zip(rows, labels):
        if int(label) == -1:
            noise += 1
        else:
            grouped[int(label)].append(row)

    zones = []
    for label, members in grouped.items():
        speeds = [float(row["speed"]) for row in members if row.get("speed") is not None]
        zones.append({
            "zona_id": label,
            "latitud": round(sum(float(row["latitude"]) for row in members) / len(members), 6),
            "longitud": round(sum(float(row["longitude"]) for row in members) / len(members), 6),
            "reportes": len(members),
            "vehiculos": len({str(row["asset_id"]) for row in members}),
            "velocidad_promedio": round(sum(speeds) / len(speeds), 2) if speeds else None,
        })
    zones.sort(key=lambda zone: (-zone["reportes"], zone["zona_id"]))
    metrics = {
        "reportes_entrenamiento": len(rows),
        "vehiculos": len({str(row["asset_id"]) for row in rows}),
        "zonas_detectadas": len(zones),
        "puntos_ruido": noise,
        "cobertura_pct": round((len(rows) - noise) * 100 / len(rows), 2),
        "radio_km": float(radius_km),
        "min_muestras": int(min_samples),
        "alcance": "Clustering geoespacial no supervisado; identifica zonas frecuentes, no rutas etiquetadas.",
    }
    return metrics, zones[:250]


# Mismos defaults que TELEMETRY_MODEL_RADIUS_KM / TELEMETRY_MODEL_MIN_SAMPLES en compose.yaml
muestra_dbscan = df.sample(min(MAX_ROWS_DBSCAN, len(df)), random_state=0).to_dict(orient="records")
metrics, zones = train_hotspots(muestra_dbscan, radius_km=0.8, min_samples=5)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

## Paso 10 — Interpretación de resultados: zonas detectadas

In [ ]:
zonas_df = pd.DataFrame(zones)
zonas_df.head(20)

## Paso 10 (cont.) — Interpretación de resultados: visualización (scatter simple; cambia radius_km / min_samples arriba y vuelve a correr para comparar)

## Paso 10 (cont.) — Interpretación de resultados: geocercas de las faenas sobre el mapa

Lee la tabla `faenas` directo de MariaDB (misma conexión del Paso 2, sin CSV intermedio) y dibuja cada faena como un polígono real, no un círculo aproximado.

In [ ]:
def load_env(path):
    """Parser minimo de .env (KEY=VALUE), sin dependencias extra."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env = load_env(Path("..") / ".env")  # credenciales de MariaDB, solo para esta seccion (faenas)

In [ ]:
import folium

with pymysql.connect(
    host="127.0.0.1", port=3306,
    user=env["MARIADB_USER"], password=env["MARIADB_PASSWORD"], database=env["MARIADB_DATABASE"],
    charset="utf8mb4", cursorclass=pymysql.cursors.DictCursor,
) as faenas_connection, faenas_connection.cursor() as faenas_cursor:
    faenas_cursor.execute(
        "SELECT id, nombre, area_m2, ST_AsText(geocerca) AS poligono_wkt FROM faenas ORDER BY id"
    )
    faenas = faenas_cursor.fetchall()

centro_faenas_lat = []
centro_faenas_lon = []
for faena in faenas:
    puntos_lonlat = faena["poligono_wkt"][len("POLYGON(("):-2].split(",")
    puntos = [(float(lat), float(lon)) for lon, lat in
              (par.strip().split(" ") for par in puntos_lonlat)]
    centro_faenas_lat.append(sum(p[0] for p in puntos) / len(puntos))
    centro_faenas_lon.append(sum(p[1] for p in puntos) / len(puntos))

mapa = folium.Map(
    location=[sum(centro_faenas_lat) / len(centro_faenas_lat), sum(centro_faenas_lon) / len(centro_faenas_lon)],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
)

for faena in faenas:
    # 'POLYGON((lon lat,lon lat,...))' -> lista de puntos (lat, lon) en orden
    puntos_lonlat = faena["poligono_wkt"][len("POLYGON(("):-2].split(",")
    puntos = [(float(lat), float(lon)) for lon, lat in
              (par.strip().split(" ") for par in puntos_lonlat)]
    etiqueta = f"{faena['nombre']}<br>area: {faena['area_m2']:.0f} m2"
    folium.Polygon(
        locations=puntos,
        color="green",
        weight=2,
        fill=True,
        fill_opacity=0.2,
        popup=etiqueta,
    ).add_to(mapa)
    # El poligono de la faena es chico frente a la separacion entre faenas (Calama y
    # Lampa quedan a ~1.200 km); a zoom_start=6 se vuelve casi invisible. El marcador
    # mantiene el mismo tamano en pantalla sin importar el zoom.
    centro_faena_lat = sum(p[0] for p in puntos) / len(puntos)
    centro_faena_lon = sum(p[1] for p in puntos) / len(puntos)
    folium.Marker(
        [centro_faena_lat, centro_faena_lon],
        tooltip=faena["nombre"],
        popup=etiqueta,
        icon=folium.Icon(color="green", icon="industry", prefix="fa"),
    ).add_to(mapa)

mapa

## Paso 11 — Despliegue: pendiente hasta terminar de validar el notebook

Esto todavía **no está listo para portar** — falta terminar y validar bien todo lo del colab primero (limpieza, ruta principal, matching con faenas, deteccion de desvios). Cuando esté todo bueno acá, se decide qué y cómo se lleva a `modelo_machine/` — no tiene que ser una copia literal del código, solo la lógica ya probada.

Notas para cuando llegue ese momento:
- `radius_km` / `min_samples` del DBSCAN (Paso 7) hay que ajustarlos hasta que las zonas tengan sentido para la operación real.
- Si se porta algo, ajustar también las variables de entorno relevantes en `compose.yaml` (`TELEMETRY_MODEL_RADIUS_KM`, `TELEMETRY_MODEL_MIN_SAMPLES`, `TELEMETRY_MODEL_MAX_ROWS`).

## Extra — Ruta principal (grilla espacial)

Objetivo distinto al DBSCAN de arriba: no buscamos "zonas frecuentes" (puntos donde se para/pasa mucho), sino **el corredor por el que circulan los camiones** entre las faenas.

**Enfoque:** dividir el mapa en una grilla de celdas chicas (grados de lat/lon), y por cada celda contar cuántos camiones *distintos* pasaron por ahí. Las celdas que forman la ruta real van a tener casi todos los camiones (40/40, o cerca); el ruido/desvíos aislados van a tener pocos. Es vectorizado con pandas (`groupby`), corre en ~1 segundo sobre los 733.005 datos — reutiliza el mismo `df` del Paso 2 (los 40 camiones), no vuelve a leer el CSV.

In [ ]:
total_camiones = df["asset_id"].nunique()
print(f"{len(df):,} reportes, {total_camiones} camiones (mismo df del Paso 2)")

TAMANO_CELDA = 0.02  # grados (~2.2 km en esta latitud); mas chico = mas preciso pero mas celdas


def construir_grilla(df, tamano_celda=TAMANO_CELDA):
    """Asigna cada reporte a una celda de la grilla (redondeo de lat/lon)."""
    df = df.copy()
    df["celda_lat"] = (df["latitude"] / tamano_celda).round().astype(int)
    df["celda_lon"] = (df["longitude"] / tamano_celda).round().astype(int)
    df["celda_id"] = df["celda_lat"].astype(str) + "_" + df["celda_lon"].astype(str)
    return df


def resumen_por_celda(df):
    """Agrupa por celda: cuantos camiones distintos y reportes tiene cada una."""
    resumen = df.groupby("celda_id").agg(
        reportes=("report_id", "count"),
        camiones=("asset_id", "nunique"),
        latitud=("latitude", "mean"),
        longitud=("longitude", "mean"),
        velocidad_promedio=("speed", "mean"),
    ).reset_index()
    return resumen.sort_values("camiones", ascending=False).reset_index(drop=True)


df_grilla = construir_grilla(df)
celdas = resumen_por_celda(df_grilla)
print(f"{len(celdas)} celdas totales")
celdas.head(20)

### Definir la "ruta principal" con los 40 camiones — hallazgo

Umbral: una celda es parte de la ruta principal si pasaron al menos el **70% de los camiones** por ahí. Resultado real con los 40 camiones juntos: la ruta principal **no llega hasta Lampa**, se queda toda en la zona de Calama/Antofagasta (lat -24 a -23). Cerca de Lampa (lat -34 a -32) ninguna celda supera los 12/40 camiones (30%).

**Conclusión:** la mayoría de los 40 camiones opera local en la zona de Calama; solo un subgrupo hace el viaje largo hasta Lampa. Por eso el umbral del 70% sobre los 40 nunca alcanza a cubrir el trayecto completo — no es un error, es lo que dicen los datos.

In [ ]:
UMBRAL_RUTA_PRINCIPAL = 0.7  # fraccion de camiones que deben pasar por la celda

minimo_camiones = int(total_camiones * UMBRAL_RUTA_PRINCIPAL)
ruta_principal = celdas[celdas["camiones"] >= minimo_camiones].reset_index(drop=True)
print(f"{len(ruta_principal)} de {len(celdas)} celdas son ruta principal "
      f"(>= {minimo_camiones} de {total_camiones} camiones)")

mapa_ruta = folium.Map(
    location=[ruta_principal["latitud"].mean(), ruta_principal["longitud"].mean()],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
)

# Ruta principal: circulos azules
for _, celda in ruta_principal.iterrows():
    folium.CircleMarker(
        [celda["latitud"], celda["longitud"]],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.6,
        popup=f"{celda['camiones']}/{total_camiones} camiones<br>{celda['reportes']} reportes<br>"
              f"vel. promedio: {celda['velocidad_promedio']:.1f} km/h",
    ).add_to(mapa_ruta)

# Resto de las celdas (fuera de la ruta principal): puntos chicos grises, para ver el contraste
resto = celdas[celdas["camiones"] < minimo_camiones]
for _, celda in resto.sample(min(500, len(resto)), random_state=0).iterrows():
    folium.CircleMarker(
        [celda["latitud"], celda["longitud"]],
        radius=1, color="gray", opacity=0.3, fill=True,
    ).add_to(mapa_ruta)

# Faenas (usa la variable `faenas` que ya trajo la celda del Paso 10 cont. de mas arriba)
for faena in faenas:
    puntos_lonlat = faena["poligono_wkt"][len("POLYGON(("):-2].split(",")
    puntos = [(float(lat), float(lon)) for lon, lat in
              (par.strip().split(" ") for par in puntos_lonlat)]
    centro_faena_lat = sum(p[0] for p in puntos) / len(puntos)
    centro_faena_lon = sum(p[1] for p in puntos) / len(puntos)
    folium.Marker(
        [centro_faena_lat, centro_faena_lon],
        tooltip=faena["nombre"],
        icon=folium.Icon(color="green", icon="industry", prefix="fa"),
    ).add_to(mapa_ruta)

mapa_ruta

### Aislar los camiones de ruta larga (los que sí llegan a Lampa)

Filtramos a los camiones con al menos un reporte cerca de Lampa (lat entre -34 y -32), y repetimos la grilla **solo con ellos**. Como ahora todos comparten el mismo trayecto largo, el umbral del 70% sí debería cubrir el corredor completo.

In [ ]:
cerca_lampa = df[(df["latitude"] >= -34) & (df["latitude"] <= -32)]
camiones_ruta_larga = sorted(cerca_lampa["asset_id"].unique())
print(f"{len(camiones_ruta_larga)} camiones de ruta larga:", camiones_ruta_larga)

df_larga = df[df["asset_id"].isin(camiones_ruta_larga)].reset_index(drop=True)
total_larga = len(camiones_ruta_larga)

df_larga_grilla = construir_grilla(df_larga)
celdas_larga = resumen_por_celda(df_larga_grilla)
print(f"{len(celdas_larga)} celdas (solo ruta larga)")

for umbral in (0.7, 0.5, 0.3):
    minimo = int(total_larga * umbral)
    n = (celdas_larga["camiones"] >= minimo).sum()
    print(f"  umbral {umbral}: {n} celdas (>= {minimo}/{total_larga})")

### Ordenar las celdas en una línea (no puntos sueltos) y dibujar la ruta

Las celdas de la ruta principal no vienen ordenadas — hay que encadenarlas en una secuencia continua para que se vea como una línea real, no una nube de puntos. Se ordenan por **vecino más cercano**: se parte de la celda más próxima a la faena del norte (Calama), y en cada paso se salta a la celda no visitada más cercana a la actual.

In [ ]:
def centro_faena(faena):
    """Centroide del poligono WKT de una faena: devuelve (lat, lon)."""
    puntos_lonlat = faena["poligono_wkt"][len("POLYGON(("):-2].split(",")
    puntos = [(float(lat), float(lon)) for lon, lat in
              (par.strip().split(" ") for par in puntos_lonlat)]
    return sum(p[0] for p in puntos) / len(puntos), sum(p[1] for p in puntos) / len(puntos)


def ordenar_ruta(celdas_ruta, origen_lat, origen_lon):
    """Encadena las celdas por vecino mas cercano, empezando por la mas proxima al origen."""
    pendientes = celdas_ruta.copy().reset_index(drop=True)
    pendientes["dist_origen"] = (pendientes["latitud"] - origen_lat) ** 2 + (pendientes["longitud"] - origen_lon) ** 2
    idx = pendientes["dist_origen"].idxmin()
    actual = pendientes.loc[idx]
    orden = [actual]
    pendientes = pendientes.drop(idx)
    while len(pendientes):
        distancias = (pendientes["latitud"] - actual["latitud"]) ** 2 + (pendientes["longitud"] - actual["longitud"]) ** 2
        idx = distancias.idxmin()
        actual = pendientes.loc[idx]
        orden.append(actual)
        pendientes = pendientes.drop(idx)
    return pd.DataFrame(orden).reset_index(drop=True)


UMBRAL_RUTA_LARGA = 0.7
minimo_larga = int(total_larga * UMBRAL_RUTA_LARGA)
ruta_larga = celdas_larga[celdas_larga["camiones"] >= minimo_larga].reset_index(drop=True)
print(f"{len(ruta_larga)} celdas de ruta principal (ruta larga, >= {minimo_larga}/{total_larga})")

faena_norte = max(faenas, key=lambda f: centro_faena(f)[0])  # lat MENOS negativa = mas al norte
origen_lat, origen_lon = centro_faena(faena_norte)
print("Punto de partida (faena norte):", faena_norte["nombre"])

ruta_ordenada = ordenar_ruta(ruta_larga, origen_lat, origen_lon)

mapa_ruta_larga = folium.Map(
    location=[ruta_larga["latitud"].mean(), ruta_larga["longitud"].mean()],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
)

folium.PolyLine(
    locations=list(zip(ruta_ordenada["latitud"], ruta_ordenada["longitud"])),
    color="blue", weight=4, opacity=0.8, tooltip="Ruta principal (Calama - Lampa)",
).add_to(mapa_ruta_larga)

for faena in faenas:
    lat, lon = centro_faena(faena)
    folium.Marker(
        [lat, lon], tooltip=faena["nombre"],
        icon=folium.Icon(color="green", icon="industry", prefix="fa"),
    ).add_to(mapa_ruta_larga)

mapa_ruta_larga